# Athena Research Operations Testing Notebook

This notebook systematically tests all operations in the Athena Research library, including:
- Basic operations
- Histogram operations
- Profile operations
- Spectral operations
- Structure function operations
- Gradient, divergence, and curl operations

In [ ]:
%load_ext autoreload
%autoreload 2
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Check if GPU is available
try:
    import cupy as cp
    print("CUDA available. Using GPU for calculations.")
    print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
    print(f"Memory: {cp.cuda.runtime.memGetInfo()[1]/1e9:.2f} GB total")
    cupy_enabled = True
    
    # Define asnumpy for CuPy arrays
    def asnumpy(a):
        """Convert CuPy array to NumPy array if necessary."""
        if isinstance(a, cp.ndarray):
            return a.get()
        return a
except ImportError:
    print("CUDA not available. Using CPU for calculations.")
    def asnumpy(a):
        """Return NumPy array unchanged."""
        return a

## 1. Loading Data

First, let's load an Athenak data file to test our operations on.

In [ ]:
from athena_research.core.utils import load

# Replace with the path to your athdf file, prefix and varnames
filedir = "/path/to/your/athdf/files/"
filenumber = 50
prefix='Turb'
varnames='.hydro_w.'
hdf_suffix='.athdf'
# Example filename
hdf_filename=prefix+varnames+str(filenumber).rjust(5, '0')+hdf_suffix
filename = filedir+hdf_filename
try:
    # Load the data file
    ad = load(filename)
    print(f"Successfully loaded {filename}")
    print(f"Time: {ad.time}")
    print(f"Mesh dimensions: {ad.Nx1} x {ad.Nx2} x {ad.Nx3}")
    print(f"Number of meshblocks: {ad.n_mbs}")
    print(f"Available variables: {list(ad.data_raw.keys())}")
except FileNotFoundError:
    print(f"File not found: {filename}")
except Exception as e:
    print(f"Error loading file: {e}")

## 2. Configure Athena Data

Let's make sure our data is properly configured before running operations.

## 3. Testing Basic Operations

Now let's test the basic operations: `calc_data`, `calc_min`, `calc_max`, `calc_sum`, `calc_avg`

In [ ]:
# Import basic operations
from athena_research.operations.basic_operations import calc_data, calc_min, calc_max, calc_sum, calc_avg

# Test calc_data
print("Testing calc_data...")
density_data = calc_data(ad, ['dens'])
print(f"Density data shape: {density_data['dens'].shape}")

# Test calc_min
print("\nTesting calc_min...")
min_vals = calc_min(ad, ['dens', 'velx', 'vely', 'velz'])
for var, val in min_vals.items():
    print(f"Minimum {var}: {val}")

# Test calc_max
print("\nTesting calc_max...")
max_vals = calc_max(ad, ['dens', 'velx', 'vely', 'velz'])
for var, val in max_vals.items():
    print(f"Maximum {var}: {val}")

# Test calc_sum
print("\nTesting calc_sum...")
sum_vals = calc_sum(ad, ['dens', 'velx', 'vely', 'velz'], weights='ones')
for var, val in sum_vals.items():
    print(f"Sum of {var}: {val}")

# Test calc_avg
print("\nTesting calc_avg...")
avg_vals = calc_avg(ad, ['dens', 'velx', 'vely', 'velz'], weights='ones')
for var, val in avg_vals.items():
    print(f"Average {var}: {val}")

## 4. Testing Histogram Operations

Next, let's test histogram operations: `set_dist` and `set_dist2d`

In [ ]:
# Import histogram functions
from athena_research.operations.histograms import set_dist, set_dist2d
import matplotlib.colors as colors

# Test 1D histogram
print("Testing set_dist...")
dist = set_dist(ad, varl=['dens'], bins=500, weights='vol', redo=True)

# Plot the distribution
plt.figure(figsize=(10, 6))
plt.plot(dist['dens']['loc'], dist['dens']['dat'])
plt.xlabel('log Density')
plt.ylabel('Frequency')
plt.yscale('log')
plt.title('Density Distribution')
plt.grid(True)
plt.show()


In [ ]:

# Test 2D histogram
print("\nTesting set_dist2d...")
var1name = 'dens'
var2name = 'mach'
dist2d = set_dist2d(ad, varl2d=[[var1name,var2name]], bins=200, weights='vol', redo=True)
try:
    var1name = 'dens'
    var2name = 'mach'
    dist2d = set_dist2d(ad, varl2d=[[var1name,var2name]], bins=200, weights='vol', redo=True)

    # Plot the 2D distribution
    plt.figure(figsize=(10, 8))
    var1 = (dist2d[var1name+'_'+var2name]['loc1'][1:] + dist2d[var1name+'_'+var2name]['loc1'][:-1]) / 2.
    var2 = (dist2d[var1name+'_'+var2name]['loc2'][1:] + dist2d[var1name+'_'+var2name]['loc2'][:-1]) / 2.
    var1_2d, var2_2d = np.meshgrid(var1, var2)
    pdf_2d = dist2d[var1name+'_'+var2name]['dat'].T
    plt.pcolormesh(var1_2d, var2_2d, pdf_2d, 
                norm=colors.LogNorm(vmin=pdf_2d.min()+1e-10, vmax=pdf_2d.max()),
                cmap='viridis', shading='auto')
    plt.colorbar(label='Frequency')
    plt.xlabel('log '+var1name)
    plt.ylabel('log '+var2name)
    plt.title('2D Distribution')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error in set_dist2d: {e}")

## 5. Testing Profile Operations

Now let's test profile operations: `set_radial` and `set_vertical`

In [ ]:
# Test vertical profile calculation
print("Testing set_vertical...")
try:
    vert = ad.vertical_profile_func(ad, varl=['dens','velx','vely','velz'], redo=True)
    
    # Plot vertical profiles
    plt.figure(figsize=(12, 6))
    plt.plot(vert['dens']['z'], vert['dens']['profile'], label='Density')
    plt.xlabel('Height (z)')
    plt.ylabel('Value')
    plt.title('Vertical Density Profile')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    # Plot vertical velocity profiles
    plt.figure(figsize=(12, 6))
    plt.plot(vert['velx']['z'], vert['velx']['profile'], label='Vx', color='blue')
    plt.plot(vert['vely']['z'], vert['vely']['profile'], label='Vy', color='green')
    plt.plot(vert['velz']['z'], vert['velz']['profile'], label='Vz', color='red')
    plt.xlabel('Height (z)')
    plt.ylabel('Velocity')
    plt.title('Vertical Velocity Profiles')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    print("Vertical profile calculation successful")
except Exception as e:
    print(f"Error in set_vertical: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# Test radial profile calculation
print("\nTesting set_radial...")
try:
    rad = ad.radial_profile_func(ad, varl=['dens', 'pres'], redo=True)
    
    # Plot radial profiles
    plt.figure(figsize=(12, 6))
    plt.plot(rad['dens']['r'], rad['dens']['profile'], label='Density')
    plt.xlabel('Radius')
    plt.ylabel('Density')
    plt.title('Radial Density Profile')
    plt.xscale('log')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    # Plot pressure profile
    plt.figure(figsize=(12, 6))
    plt.plot(rad['pres']['r'], rad['pres']['profile'], label='Pressure', color='orange')
    plt.xlabel('Radius')
    plt.ylabel('Pressure')
    plt.title('Radial Pressure Profile')
    plt.xscale('log')
    plt.grid(True)
    plt.legend()
    plt.show()
    
    print("Radial profile calculation successful")
except Exception as e:
    print(f"Error in set_radial: {e}")
    import traceback
    traceback.print_exc()

## 6. Testing Gradient, Divergence, and Curl Operations

Let's test the gradient, divergence, and curl operations.

In [ ]:
# Import gradient, divergence, and curl operations
from athena_research.operations.grad_div_curl import gradient, divergence, curl

# Test gradient operation
print("Testing gradient operation...")
try:
    grad_dens = gradient(ad, 'dens')
    print(f"Gradient shape: {[g.shape for g in grad_dens]}")
    print(f"Gradient components min/max: \n" +
          f"  x: {asnumpy(grad_dens[0]).min():.3e} to {asnumpy(grad_dens[0]).max():.3e}\n" +
          f"  y: {asnumpy(grad_dens[1]).min():.3e} to {asnumpy(grad_dens[1]).max():.3e}\n" +
          f"  z: {asnumpy(grad_dens[2]).min():.3e} to {asnumpy(grad_dens[2]).max():.3e}")
    
    # Visualize a slice of the gradient magnitude
except Exception as e:
    print(f"Error in gradient calculation: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# Test divergence operation
print("\nTesting divergence operation...")
try:
    div_vel = divergence(ad, 'velx', 'vely', 'velz')
    print(f"Divergence shape: {div_vel.shape}")
    print(f"Divergence min/max: {asnumpy(div_vel).min():.3e} to {asnumpy(div_vel).max():.3e}")
    
except Exception as e:
    print(f"Error in divergence calculation: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# Test curl operation
print("\nTesting curl operation...")
try:
    curl_vel = curl(ad, 'velx', 'vely', 'velz', simultaneous_blocks=10)
    print(f"Curl shape: {[c.shape for c in curl_vel]}")
    print(f"Curl components min/max: \n" +
          f"  x: {asnumpy(curl_vel[0]).min():.3e} to {asnumpy(curl_vel[0]).max():.3e}\n" +
          f"  y: {asnumpy(curl_vel[1]).min():.3e} to {asnumpy(curl_vel[1]).max():.3e}\n" +
          f"  z: {asnumpy(curl_vel[2]).min():.3e} to {asnumpy(curl_vel[2]).max():.3e}")
    
except Exception as e:
    print(f"Error in curl calculation with simultaneous_blocks=1: {e}")
    
    # If still failing, try releasing memory first
    import gc
    print("Attempting to free memory and retry with auto_select=False...")
    gc.collect()
    if 'cupy_enabled' in globals() and cupy_enabled:
        import cupy as cp
        cp.get_default_memory_pool().free_all_blocks()
        print(f"GPU memory after cleanup: {cp.cuda.runtime.memGetInfo()[0]/1e9:.2f}GB free / {cp.cuda.runtime.memGetInfo()[1]/1e9:.2f}GB total")
    
    try:
        # Try using auto_select=False to use the most memory-efficient method
        curl_vel = curl(ad, 'velx', 'vely', 'velz', auto_select=False)
        print(f"Curl shape: {[c.shape for c in curl_vel]}")
        print(f"Curl components min/max: \n" +
              f"  x: {asnumpy(curl_vel[0]).min():.3e} to {asnumpy(curl_vel[0]).max():.3e}\n" +
              f"  y: {asnumpy(curl_vel[1]).min():.3e} to {asnumpy(curl_vel[1]).max():.3e}\n" +
              f"  z: {asnumpy(curl_vel[2]).min():.3e} to {asnumpy(curl_vel[2]).max():.3e}")
    except Exception as e2:
        print(f"Still failing with auto_select=False: {e2}")
        import traceback
        traceback.print_exc()

## 7. Testing Spectral Operations

Let's test the spectral analysis operations: `set_spectrum` and `set_spectrum_helmholtz`

In [ ]:
# Import spectral operations
from athena_research.operations.spectra import set_spectrum, set_spectrum_helmholtz

# Test standard power spectrum
print("Testing set_spectrum...")
try:
    spectra = set_spectrum(ad=ad, varl=['velx', 'vely', 'velz'], strat_flag=False, redo=True)
    
    # Extract data for plotting
    k = spectra['velx']['k']
    velx_ps = spectra['velx']['spectrum']
    vely_ps = spectra['vely']['spectrum']
    velz_ps = spectra['velz']['spectrum']
    total_ps = velx_ps + vely_ps + velz_ps
    
    # Plot power spectra
    plt.figure(figsize=(10, 6))
    plt.loglog(k, velx_ps, label='Vx')
    plt.loglog(k, vely_ps, label='Vy')
    plt.loglog(k, velz_ps, label='Vz')
    plt.loglog(k, total_ps, label='Total', linewidth=2, color='black')
    
    # Add Kolmogorov scaling for reference
    k_range = np.logspace(np.log10(k[5]), np.log10(k[-10]), 100)
    plt.loglog(k_range, 0.01*(k_range/k_range[0])**(-5/3), '--', label=r"$k^{-5/3}$ (K41)")
    
    plt.xlabel('k')
    plt.ylabel('P(k)')
    plt.title('Velocity Power Spectra')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Standard spectrum calculation successful")
except Exception as e:
    print(f"Error in set_spectrum: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# Test Helmholtz decomposed power spectrum
print("\nTesting set_spectrum_helmholtz...")
try:
    spectra_helmholtz = set_spectrum_helmholtz(ad=ad, var='vel', strat_flag=False, redo=True)
    
    # Extract data for plotting
    k_h = spectra_helmholtz['vel']['k']
    sol_ps = spectra_helmholtz['vel']['spectrum_sol']  # Solenoidal (incompressible) component
    comp_ps = spectra_helmholtz['vel']['spectrum_comp']  # Compressible component
    total_ps_h = sol_ps + comp_ps
    
    # Plot decomposed power spectrum
    plt.figure(figsize=(10, 6))
    plt.loglog(k_h, sol_ps, label='Solenoidal (∇·v=0)', linewidth=2)
    plt.loglog(k_h, comp_ps, label='Compressible (∇×v=0)', linewidth=2)
    plt.loglog(k_h, total_ps_h, label='Total', linewidth=2, color='black', alpha=0.7)
    
    # Add Kolmogorov scaling for reference
    k_range = np.logspace(np.log10(k_h[5]), np.log10(k_h[-10]), 100)
    plt.loglog(k_range, 0.01*(k_range/k_range[0])**(-5/3), '--', label=r"$k^{-5/3}$ (K41)")
    
    plt.xlabel('k')
    plt.ylabel('P(k)')
    plt.title('Helmholtz Decomposed Velocity Power Spectrum')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Compare to standard spectrum
    if 'spectra' in locals():
        plt.figure(figsize=(10, 6))
        plt.loglog(k, total_ps, label='Standard Method')
        plt.loglog(k_h, total_ps_h, label='Helmholtz Method', linestyle='--')
        plt.xlabel('k')
        plt.ylabel('P(k)')
        plt.title('Comparison of Spectrum Methods')
        plt.legend()
        plt.grid(True, which='both', ls='--', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    print("Helmholtz spectrum calculation successful")
except Exception as e:
    print(f"Error in set_spectrum_helmholtz: {e}")
    import traceback
    traceback.print_exc()

## 8. Testing Structure Function Operations

Finally, let's test the structure function operations: `set_sf` and `set_sf_helmholtz`

In [ ]:
# Import structure function operations
from athena_research.operations.structure_functions import set_sf, set_sf_helmholtz

# Test structure function calculation
print("Testing set_sf...")
try:
    # Calculate structure functions for velocity
    sf = set_sf(
        ad=ad,
        varl=['velx', 'vely', 'velz'],
        weights='ones',
        max_order=3,
        npairs=1e8,  # Reduced number for quicker testing
        nbins=100,
        log_bin_flag=False,
        redo=True,
        auto_select=False
    )
    
    # Plot SF2 only for clarity
    plt.figure(figsize=(10, 6))
    for var, color in zip(['velx', 'vely', 'velz'], ['blue', 'green', 'red']):
        if var in sf:
            r_bins = sf[var]['r']
            sf2 = sf[var]['sf'][1] / sf[var]['num_bin_points']
            plt.loglog(r_bins, sf2, '-o', label=f"{var} SF-2", color=color)
    # Add Kolmogorov scaling for SF2
    r_range = np.logspace(np.log10(r_bins[5]), np.log10(r_bins[-10]), 100)
    plt.xlabel('r')
    plt.ylabel('SF(r)')
    # Add Kolmogorov scaling for SF2
    plt.loglog(r_range, 5e-4*(r_range/r_range[0])**(2/3), '--k', linewidth=2, label=r"$r^{2/3}$ (K41 for SF2)")
    plt.title('Second-Order Velocity Structure Functions')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Structure function calculation successful")
except Exception as e:
    print(f"Error in set_sf: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# Test Helmholtz decomposed structure functions
print("\nTesting set_sf_helmholtz...")
try:
    sf_helmholtz = set_sf_helmholtz(
        ad=ad,
        var='vel',
        weights='ones',
        max_order=3,
        npairs=1e8,  # Reduced number for quicker testing
        nbins=100,
        log_bin_flag=False,
        redo=True
    )
    
    # Plot Helmholtz decomposed structure functions
    plt.figure(figsize=(10, 6))
    
    r_bins = sf_helmholtz['vel']['r']
    sf2_sol = sf_helmholtz['vel']['sf_sol'][1] / sf_helmholtz['vel']['num_bin_points']
    sf2_comp = sf_helmholtz['vel']['sf_comp'][1] / sf_helmholtz['vel']['num_bin_points']
    sf2_total = sf2_sol + sf2_comp
    
    plt.loglog(r_bins, sf2_sol, '-o', label='Solenoidal (∇·v=0)', linewidth=2)
    plt.loglog(r_bins, sf2_comp, '-o', label='Compressible (∇×v=0)', linewidth=2)
    plt.loglog(r_bins, sf2_total, '-o', label='Total', linewidth=2, color='black')
    
    # Add Kolmogorov scaling
    r_range = np.logspace(np.log10(r_bins[5]), np.log10(r_bins[-10]), 100)
    plt.loglog(r_range, 5e-4*(r_range/r_range[0])**(2/3), '--k', linewidth=2, label=r"$r^{2/3}$ (K41 for SF2)")
    
    plt.xlabel('r')
    plt.ylabel('SF2(r)')
    plt.title('Helmholtz Decomposed Structure Functions')
    plt.legend()
    plt.grid(True, which='both', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Helmholtz structure function calculation successful")
except Exception as e:
    print(f"Error in set_sf_helmholtz: {e}")
    import traceback
    traceback.print_exc()

## Summary

We have successfully tested all the operations available in the Athena Research library:

1. Basic Operations:
   - `calc_data`: Retrieves data arrays
   - `calc_min`, `calc_max`: Compute min/max values
   - `calc_sum`, `calc_avg`: Calculate sums and averages

2. Histogram Operations:
   - `set_dist`: Creates 1D histograms
   - `set_dist2d`: Creates 2D histograms

3. Profile Operations:
   - `set_vertical`: Calculates vertical profiles
   - `set_radial`: Calculates radial profiles

4. Gradient Operations:
   - `gradient`: Calculates spatial gradients
   - `divergence`: Calculates vector field divergence
   - `curl`: Calculates vector field curl

5. Spectral Operations:
   - `set_spectrum`: Calculates power spectra
   - `set_spectrum_helmholtz`: Calculates Helmholtz-decomposed spectra

6. Structure Function Operations:
   - `set_sf`: Calculates structure functions
   - `set_sf_helmholtz`: Calculates Helmholtz-decomposed structure functions

These operations provide a comprehensive toolkit for analyzing Athenak simulation data.